\begin{align*}
\phi_{i|y=1} = p(x_i = 1|y = 1)\\
\phi_{i|y=0} = p(x_i =1|y = 0)\\ 
\phi_y = p(y = 1)\\
\phi_{j|y=1} = \frac{\sum_{i=1}^{m} x^{(i)}_j 1\{y^{(i)} = 1\} + 1}{\sum_{i=1}^{m} \sum_{l=1}^{n} x^{(i)}_l 1\{y^{(i)} = 1\} + n}\\
\phi_{j|y=0} = \frac{\sum_{i=1}^{m} x^{(i)}_j 1\{y^{(i)} = 0\} + 1}{\sum_{i=1}^{m} \sum_{l=1}^{n} x^{(i)}_l 1\{y^{(i)} = 0\} + n}\\
\phi_y = \frac{\sum_{i=1}^{m}1\{y^{(i)} = 1\}}{m}\\
p(y = 1|x) = \frac{p(x|y = 1)p(y = 1)}{p(x)}\\
p(y = 0|x) = \frac{p(x|y = 0)p(y = 0)}{p(x)}\\
\end{align*}
To make prediction we pick label which has bigger value of $p(y|x)$ since both terms have $p(x)$ we can omit it.

In [1]:
import numpy as np
import pandas as pd

In [2]:
def get_words(message):
    #Get the normalized list of words from a message string.

    return message.lower().split(' ')


def create_dictionary(messages):
#Create a dictionary mapping words to integer indices.

    word_count={}
    for message in messages:
        words=get_words(message)
        for word in words:
            word_count[word]=word_count.get(word,0)+1
    word_dictionary={}
    index=0
    for word,count in word_count.items():
        if count>=5:
            word_dictionary[word]=index
            index+=1
    return word_dictionary



def transform_text(messages, word_dictionary):
    #Transform a list of text messages into a numpy array for further processing.
    array=[]
    for message in messages:
        words=get_words(message)
        row=np.zeros(len(word_dictionary))
        for word in words:
            index=word_dictionary.get(word,-1)
            if index!=-1:
                row[index]+=1
        array.append(row)
    return np.array(array)


In [3]:
df=pd.read_csv('data/spam.csv',encoding_errors='ignore')

In [4]:
X,y=df['v2'],df['v1']

In [5]:
y=y.map({'ham':0,'spam':1})

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [7]:
dictionary=create_dictionary(X_train.values)

In [8]:
train_matrix = transform_text(X_train.values, dictionary)
test_matrix = transform_text(X_test.values, dictionary)

In [9]:
class NaiveBayes:
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        y1_count=np.sum(X[y==1])
        y0_count=np.sum(X[y==0])
        self.p_y1=len(y[y==1])/m
        self.p_y0=len(y[y==0])/m
        self.p_xy1=(np.sum(X[y==1],axis=0)+1)/(y1_count+n)
        self.p_xy0=(np.sum(X[y==0],axis=0)+1)/(y0_count+n)
    def predict(self,X):
        X=np.asarray(X)
        y1=np.sum(X*np.log(self.p_xy1),axis=1)+np.log(self.p_y1)#mx1
        y0=np.sum(X*np.log(self.p_xy0),axis=1)+np.log(self.p_y0)#mx1
        return np.argmax(np.column_stack((y0, y1)),axis=1)

In [10]:
nb=NaiveBayes()
nb.fit(train_matrix,y_train)
preds=nb.predict(test_matrix)
np.mean(preds==y_test)

np.float64(0.97847533632287)

In [12]:
#Words which indicate spam mail "the most"
flipped_dict = {value: key for key, value in dictionary.items()}
[flipped_dict[i] for i in np.argsort(np.log(nb.p_xy1)-np.log(nb.p_xy0),descending=True)[:5]]

['claim', 'won', 'prize', '500', 'urgent!']